# Verificacion de la calidad de la tabla plata.prestamos

Proposito del script:  
- Verificar columna por columna la calidad de los datos.

# Estableciendo la Conexion

In [1]:
# Importanto las librerias y creando la conexion 
import pandas as pd 
import numpy as np 
from datetime import date
from funciones import formato_clasificacion_riesgo_sbs,asignacion_estado_prestamo
from conexiones_y_rutas import obtener_engine
engine = obtener_engine()

df_prestamos = pd.read_sql(
    "SELECT * FROM plata.prestamos",
    con= engine
)

df_prestamos_tra = df_prestamos.copy()

# Archivos de Ayuda

In [2]:
df_productos_crediticios= pd.read_sql(
    "SELECT * FROM plata.productos_crediticios",
    con= engine
)
df_productos_crediticios_tra = df_productos_crediticios.copy()
df_productos_crediticios_tra.head()

,producto_id,nombre_producto,tipo_credito,tasa_nom_min,tasa_nom_max,plazo_min_meses,plazo_max_meses,monto_minimo,monto_maximo,requiere_garantia,moneda,dwh_fecha_carga
0,1,Crédito Personal Libre Disponibilidad,Personal,18.0,36.0,6,60,1000.0,50000.0,0,PEN,2026-08-03 18:46:03.850
1,2,Crédito Hipotecario Vivienda,Hipotecario,7.5,11.0,60,360,50000.0,1000000.0,1,PEN,2026-08-03 18:46:03.850
2,3,Crédito Vehicular,Vehicular,2.5,18.0,12,72,10000.0,200000.0,1,PEN,2026-08-03 18:46:03.850
3,4,Crédito MYPE Capital De Trabajo,Microempresa,20.0,48.0,6,48,2000.0,100000.0,0,PEN,2026-08-03 18:46:03.850
4,5,Crédito de Consumo,Consumo,24.0,42.0,3,36,500.0,20000.0,0,PEN,2026-08-03 18:46:03.850


In [3]:
# Cambia de Object a boolean 
df_productos_crediticios_tra["requiere_garantia"] = (
    df_productos_crediticios_tra["requiere_garantia"]
    .astype(int)
    .astype(bool)
)

In [4]:
df_sucursales = pd.read_sql(
    "SELECT sucursal_id,fecha_apertura FROM plata.sucursales",
    con= engine
)
df_sucursales_tra = df_sucursales.copy()
df_sucursales_tra.head()

,sucursal_id,fecha_apertura
0,1,2005-07-24
1,2,2010-01-03
2,3,2007-04-20
3,4,2010-02-19
4,5,2007-02-07


In [5]:
df_sucursales_tra["fecha_apertura"] = pd.to_datetime(df_sucursales_tra.fecha_apertura)

In [6]:
df_clientes = pd.read_sql(
    "SELECT * FROM plata.clientes",
    con= engine
)
df_clientes_tra = df_clientes.copy()
df_clientes_tra.head()

,cliente_id,tipo_documento,numero_documento,nombres,apellido_paterno,apellido_materno,fecha_nacimiento,edad,genero,estado_civil,...,egresos_mensuales,patrimonio_estimado,score_crediticio,segmento_cliente,canal_captacion,antiguedad_cliente_meses,fecha_registro,sucursal_id,estado_cliente,dwh_fecha_carga
0,1,DNI,60366909,Carmen,Ramírez,Flores,1977-09-16,48,Femenino,Casado,...,888.76,39631.77,550,Regular,Agencia,122,2016-05-07,14,Activo,2026-08-03 18:46:04.580
1,2,DNI,62729806,Cecilia,Cusi,Cusi,1964-03-13,62,Femenino,Casado,...,3037.05,125691.99,528,Regular,Digital,105,2017-10-18,13,Activo,2026-08-03 18:46:04.580
2,3,CE,641708053,Juan,Ortiz,Medina,1964-08-01,62,Masculino,Casado,...,2302.13,226055.14,493,Regular,Agencia,90,2019-01-08,13,Activo,2026-08-03 18:46:04.580
3,4,DNI,29912419,Carlos,Flores,Morales,1976-11-09,49,Masculino,Soltero,...,763.53,104986.09,490,Regular,Telemarketing,196,2010-03-12,15,Activo,2026-08-03 18:46:04.580
4,5,DNI,86518506,Sandra,González,Morales,1980-11-20,45,Femenino,Viudo,...,405.03,37044.25,392,Regular,Digital,162,2013-01-08,4,Activo,2026-08-03 18:46:04.580


In [7]:
columnas_fecha = [
    "fecha_nacimiento",
    "fecha_registro"
]

for columna in columnas_fecha:
    df_clientes_tra[columna] = pd.to_datetime(df_clientes_tra[columna])

In [8]:
df_oficial = pd.read_sql(
    "SELECT oficial_id, fecha_ingreso FROM plata.oficiales_credito",
    con= engine
)
df_oficial_tra = df_oficial.copy()
df_oficial_tra.head()

,oficial_id,fecha_ingreso
0,1,2017-12-20
1,2,2015-06-29
2,3,2010-09-20
3,4,2020-01-01
4,5,2020-01-09


In [9]:
df_oficial_tra["fecha_ingreso"] = pd.to_datetime(df_oficial_tra.fecha_ingreso)

# Resumen de las Columnas

- **prestamo_id**: Identificador unico de cada prestamo.  
- **cliente_id**: Identificador del cliente asociado al prestamo.  
- **sucursal_id**: Identificador de la sucursal asociada al prestamo.  
- **producto_id**: Identificador del producto asociado al prestamo.  
- **oficial_id**:  Identificador del oficial de credito asociado al prestamo.  
- **numero_contrato**: Numero de contrato del prestamo "CONT-000000(ID_PRESTAMO)" => Maximo 13 caracteres 
- **fecha_otorgamiento**: Fecha de otorgamiento del credito.   
- **fecha_vencimiento**: Fecha de vencimiento del credito.  
- **monto_original**: Monto Original del credito.  
- **saldo_capital_vigente**: Saldo Capital Vigente.   
- **tasa_interes_nominal_anual**:  Tasa nominal anual del credito.  
- **tasa_interes_efectiva_anual**:  Tasa efectiva anual del credito.  
- **plazo_meses**: Plazo en meses para cancelar el credito.  
- **tipo_credito**: Tipo de Credito (ejem: Personal, Microempresa o Hipotecario).  
- **moneda**: Moneda del credito (ejem: USD o PEN).  
- **frecuencia_pago**: Frecuencia on que se realizan los pagos (ejem: Mensual).  
- **cuota_programada**: Monto a pagar en cada cuota. 
- **numero_cuotas_total**: Numero de cuotas totales del credito.  
- **numero_cuotas_pagadas**: Numero de cuotas pagadas.  
- **numero_cuotas_pendientes**: Numero de cuotas pendientes.  
- **estado**: Estado del credito (ejem: Vigente o Moroso).    
- **dias_mora**: Dias de mora.  
- **clasificacion_riesgo_sbs**: Casificacion de riesgo de la sbs (ejem: Normal, CPP o Perdida).  
- **garantia_tipo**: Tipo de garantia entregada (ejem: Aval, Carta Fianza o Sin Garantia ).  
- **garantia_valor**: Valor de la garantia.  
- **proposito_credito**: Proposito del credito (ejem: Viaje / Turismo, Mejoras Del Hogar o Expansión De Negocio).    
- **canal_desembolso**: Canal por el cual realiza el desembolso (ejem: Transferencia Bancaria, Agencia o Cheque De Gerencia).  
- **fecha_primer_pago_programado**: Fecha del primer pago programado. 
- **fecha_ultimo_pago_real**: Fecha donde se realizo el ultimo pago del crédito.  

# Verificacion de la Calidad de Datos

In [10]:
df_prestamos_tra.info()  

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6495 entries, 0 to 6494
Data columns (total 30 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   prestamo_id                   6495 non-null   int64         
 1   cliente_id                    6495 non-null   int64         
 2   sucursal_id                   6495 non-null   int64         
 3   producto_id                   6495 non-null   int64         
 4   oficial_id                    6495 non-null   int64         
 5   numero_contrato               6495 non-null   object        
 6   fecha_otorgamiento            6495 non-null   object        
 7   fecha_vencimiento             6495 non-null   object        
 8   monto_original                6495 non-null   float64       
 9   saldo_capital_vigente         6495 non-null   float64       
 10  tasa_interes_nominal_anual    6495 non-null   float64       
 11  tasa_interes_efectiva_anual   

In [11]:
columnas_fecha = [
    "fecha_otorgamiento",
    "fecha_vencimiento",
    "fecha_primer_pago_programado",
    "fecha_ultimo_pago_real"
]

for columna in columnas_fecha:
    df_prestamos_tra[columna] = pd.to_datetime(df_prestamos_tra[columna])

In [12]:
df_prestamos_tra.head()

,prestamo_id,cliente_id,sucursal_id,producto_id,oficial_id,numero_contrato,fecha_otorgamiento,fecha_vencimiento,monto_original,saldo_capital_vigente,...,estado,dias_mora,clasificacion_riesgo_sbs,garantia_tipo,garantia_valor,proposito_credito,canal_desembolso,fecha_primer_pago_programado,fecha_ultimo_pago_real,dwh_fecha_carga
0,1,4848,23,6,60,CONT-00000001,2023-05-07,2025-10-23,9218.73,3596.20,...,Vigente,0,Normal,n/a,0.0,Viaje / Turismo,Transferencia Bancaria,2023-06-06,2024-12-27,2026-08-03 18:46:43.170
1,2,44,5,1,21,CONT-00000002,2021-02-12,2025-07-21,7767.38,1473.62,...,Vigente,0,Normal,Aval,0.0,Mejoras del Hogar,Agencia,2021-03-14,2024-12-23,2026-08-03 18:46:43.170
2,3,2474,16,1,35,CONT-00000003,2023-11-22,2027-11-01,12512.04,10640.43,...,Vigente,0,Normal,Aval,0.0,Mejoras del Hogar,Transferencia Bancaria,2023-12-22,2024-12-16,2026-08-03 18:46:43.170
3,4,637,15,4,32,CONT-00000004,2024-04-19,2024-10-16,3555.11,0.00,...,Cancelado,16,CPP,Carta Fianza,0.0,Expansión de Negocio,Agencia,2024-05-19,2024-11-01,2026-08-03 18:46:43.170
4,5,3622,10,4,12,CONT-00000005,2020-04-19,2022-10-06,8388.15,0.00,...,Cancelado,0,Normal,Sin Garantía,0.0,Capital de Trabajo,Agencia,2020-05-19,2022-10-06,2026-08-03 18:46:43.170


In [13]:
# Verifica si existen registro duplicados 
# Resultados Esperados: Tabla Vacia
df_prestamos_tra[df_prestamos_tra.duplicated(keep=False)]

,prestamo_id,cliente_id,sucursal_id,producto_id,oficial_id,numero_contrato,fecha_otorgamiento,fecha_vencimiento,monto_original,saldo_capital_vigente,...,estado,dias_mora,clasificacion_riesgo_sbs,garantia_tipo,garantia_valor,proposito_credito,canal_desembolso,fecha_primer_pago_programado,fecha_ultimo_pago_real,dwh_fecha_carga


## prestamo_id

In [14]:
# Verifica que no existan ids duplicados de prestamo 
# Resultados Esperados: Tabla Vacia 
df_prestamos_tra[df_prestamos_tra.prestamo_id.duplicated(keep=False)]

,prestamo_id,cliente_id,sucursal_id,producto_id,oficial_id,numero_contrato,fecha_otorgamiento,fecha_vencimiento,monto_original,saldo_capital_vigente,...,estado,dias_mora,clasificacion_riesgo_sbs,garantia_tipo,garantia_valor,proposito_credito,canal_desembolso,fecha_primer_pago_programado,fecha_ultimo_pago_real,dwh_fecha_carga


In [15]:
# Verifica si existen ids negativos o 0 
# Resultados Esperados: Tabla Vacia 
df_prestamos_tra[df_prestamos_tra.prestamo_id <= 0]

,prestamo_id,cliente_id,sucursal_id,producto_id,oficial_id,numero_contrato,fecha_otorgamiento,fecha_vencimiento,monto_original,saldo_capital_vigente,...,estado,dias_mora,clasificacion_riesgo_sbs,garantia_tipo,garantia_valor,proposito_credito,canal_desembolso,fecha_primer_pago_programado,fecha_ultimo_pago_real,dwh_fecha_carga


## cliente_id

In [16]:
# Verifica que cliente_id sea valido 
# Resultados Esperados: Tabla Vacia 
df_prestamos_tra[df_prestamos_tra.cliente_id <= 0]

,prestamo_id,cliente_id,sucursal_id,producto_id,oficial_id,numero_contrato,fecha_otorgamiento,fecha_vencimiento,monto_original,saldo_capital_vigente,...,estado,dias_mora,clasificacion_riesgo_sbs,garantia_tipo,garantia_valor,proposito_credito,canal_desembolso,fecha_primer_pago_programado,fecha_ultimo_pago_real,dwh_fecha_carga


In [17]:
# Verifica que cliente_id exista en la tabla de clientes 
# Resultados Esperados: both: 6495, left_only: 0, right_only: 0
verificar_cliente_id = df_prestamos_tra.merge(
    right=df_clientes_tra,
    on='cliente_id',
    how= 'left',
    indicator=True
)

verificar_cliente_id._merge.value_counts()

_merge
both          6495
left_only        0
right_only       0
Name: count, dtype: int64

## sucursal_id

In [18]:
# Verifica que sucursal_id sea valido 
# Resultados Esperados: Tabla Vacia 
df_prestamos_tra[df_prestamos_tra.sucursal_id <= 0]

,prestamo_id,cliente_id,sucursal_id,producto_id,oficial_id,numero_contrato,fecha_otorgamiento,fecha_vencimiento,monto_original,saldo_capital_vigente,...,estado,dias_mora,clasificacion_riesgo_sbs,garantia_tipo,garantia_valor,proposito_credito,canal_desembolso,fecha_primer_pago_programado,fecha_ultimo_pago_real,dwh_fecha_carga


In [19]:
# Verifica que sucursal_id exista en la tabla de sucursales 
# Resultados Esperados: both: 6495, left_only: 0, right_only: 0
verificar_sucursal_id = df_prestamos_tra.merge(
    right=df_sucursales_tra,
    on='sucursal_id',
    how='left',
    indicator=True
)

verificar_sucursal_id._merge.value_counts()

_merge
both          6495
left_only        0
right_only       0
Name: count, dtype: int64

## producto_id

In [20]:
# Verifica que producto_id sea valido 
# Resultados Esperados: Tabla Vacia 
df_prestamos_tra[df_prestamos_tra.producto_id <= 0]

,prestamo_id,cliente_id,sucursal_id,producto_id,oficial_id,numero_contrato,fecha_otorgamiento,fecha_vencimiento,monto_original,saldo_capital_vigente,...,estado,dias_mora,clasificacion_riesgo_sbs,garantia_tipo,garantia_valor,proposito_credito,canal_desembolso,fecha_primer_pago_programado,fecha_ultimo_pago_real,dwh_fecha_carga


In [21]:
# Verifica que producto_id exista en la tabla de productos 
# Resultados Esperados: both: 6495, left_only: 0, right_only: 0
verificar_producto_id = df_prestamos_tra.merge(
    right=df_productos_crediticios_tra,
    on='producto_id',
    how='left',
    indicator=True
)

verificar_producto_id._merge.value_counts()

_merge
both          6495
left_only        0
right_only       0
Name: count, dtype: int64

## oficial_id

In [22]:
# Verifica que oficial_id sea valido 
# Resultados Esperados: Tabla Vacia 
df_prestamos_tra[df_prestamos_tra.oficial_id <= 0]

,prestamo_id,cliente_id,sucursal_id,producto_id,oficial_id,numero_contrato,fecha_otorgamiento,fecha_vencimiento,monto_original,saldo_capital_vigente,...,estado,dias_mora,clasificacion_riesgo_sbs,garantia_tipo,garantia_valor,proposito_credito,canal_desembolso,fecha_primer_pago_programado,fecha_ultimo_pago_real,dwh_fecha_carga


In [23]:
# Verifica que oficial_id exista en la tabla de oficiales_credito 
# Resultados Esperados: both: 6495, left_only: 0, right_only: 0
verificar_oficial_id = df_prestamos_tra.merge(
    right= df_oficial_tra,
    on='oficial_id',
    how='left',
    indicator=True
)

verificar_oficial_id._merge.value_counts()

_merge
both          6495
left_only        0
right_only       0
Name: count, dtype: int64

## numero_contrato  

In [24]:
# Verfica que tengan el formato correcto y la extencion correcta
# Resultados Esperados: Tabla Vacia
df_prestamos_tra.numero_contrato[
    (df_prestamos_tra.numero_contrato
        != df_prestamos_tra.numero_contrato.str.strip().str.upper())
    |
    (df_prestamos_tra.numero_contrato.apply(len)!= 13)
    |
    (df_prestamos_tra.numero_contrato.isna())
]

Series([], Name: numero_contrato, dtype: object)

In [25]:
# Verifica el formato de la referencia del contrato "CONT-000000(ID_PRESTAMO)" => Maximo 13 caracteres 
# Resultados Esperados: Tabla Vacia
# 'CONT-'+ RELLENAR_CON_CEROS + ID_PRESTAMO => Maximo 13 caracteres
num_pag_gene = (
    "CONT-"
    + df_prestamos_tra["prestamo_id"]
        .astype(str)
        .str.zfill(8)
)

df_prestamos_tra[df_prestamos_tra.numero_contrato != num_pag_gene]

,prestamo_id,cliente_id,sucursal_id,producto_id,oficial_id,numero_contrato,fecha_otorgamiento,fecha_vencimiento,monto_original,saldo_capital_vigente,...,estado,dias_mora,clasificacion_riesgo_sbs,garantia_tipo,garantia_valor,proposito_credito,canal_desembolso,fecha_primer_pago_programado,fecha_ultimo_pago_real,dwh_fecha_carga


## fecha_otorgamiento 

In [26]:
# Transforma a formato fecha y cambia las fechas a nan, en caso error 
error_fecha_otor = pd.to_datetime(
    df_prestamos_tra.fecha_otorgamiento,
    errors='coerce'
)
# Muestra las fechas que generan error 
# Resultados Esperados: Tabla Vacia 
df_prestamos_tra.fecha_otorgamiento[error_fecha_otor.isna()]

Series([], Name: fecha_otorgamiento, dtype: datetime64[ns])

In [27]:
# Verifica si existen fechas de otorgamiento futuras 
# Resultados Esperados: Tabla Vacia
df_prestamos_tra[df_prestamos_tra.fecha_otorgamiento.dt.date >= date.today()]

,prestamo_id,cliente_id,sucursal_id,producto_id,oficial_id,numero_contrato,fecha_otorgamiento,fecha_vencimiento,monto_original,saldo_capital_vigente,...,estado,dias_mora,clasificacion_riesgo_sbs,garantia_tipo,garantia_valor,proposito_credito,canal_desembolso,fecha_primer_pago_programado,fecha_ultimo_pago_real,dwh_fecha_carga


## fecha_vencimiento

**Nota**: No se verifica si existen fechas de vencimiento futuras, porque existen creditos que no se han terminado de pagar

In [28]:
# Transforma a formato fecha y cambia las fechas a nan, en caso error 
error_fecha_ven = pd.to_datetime(
    df_prestamos_tra.fecha_vencimiento,
    errors='coerce'
)
# Muestra las fechas que generar error 
# Resultados Esperados: Tabla Vacia 
df_prestamos_tra.fecha_vencimiento[error_fecha_ven.isna()]

Series([], Name: fecha_vencimiento, dtype: datetime64[ns])

In [29]:
# Muesta las fechas de otorgamiento que sean mayores a las fechas de vencimiento  
# Resultados Esperados: Tabla Vacia 
df_prestamos_tra[["cliente_id","fecha_otorgamiento","fecha_vencimiento"]][
    df_prestamos_tra.fecha_otorgamiento > df_prestamos_tra.fecha_vencimiento
    ]

,cliente_id,fecha_otorgamiento,fecha_vencimiento


## monto_original 

In [30]:
# Verifica que los montos no sean negativos o cero 
# Resultados Esperados: Tabla Vacia
df_prestamos_tra.monto_original[df_prestamos_tra.monto_original <= 0]

Series([], Name: monto_original, dtype: float64)

In [31]:
# LEFT JOIN, con productos 
df_merge_veri_monto = df_prestamos_tra.merge(
    right= df_productos_crediticios_tra,
    on="producto_id",
    how='left'
)

# Verifica que los montos se encuentren en el rango de monto correspondiente para dicho producto
# Resultados Esperados: Tabla Vacia
df_merge_veri_monto[["producto_id","monto_original","monto_minimo","monto_maximo"]][
                (df_merge_veri_monto.monto_original < df_merge_veri_monto.monto_minimo) 
                | (
                    (df_merge_veri_monto.monto_maximo.notna())
                    & (df_merge_veri_monto.monto_original > df_merge_veri_monto.monto_maximo))
                ]

,producto_id,monto_original,monto_minimo,monto_maximo


## saldo_capital_vigente

In [32]:
# Verifica si los saldos son negativos
# Resultados Esperados: Tabla Vacia 
df_prestamos_tra.saldo_capital_vigente[df_prestamos_tra.saldo_capital_vigente < 0]

Series([], Name: saldo_capital_vigente, dtype: float64)

## tasa_interes_nominal_anual

In [33]:
# Verifica si la tasa_interes_nominal_anual es negativa o 0 
# Resultados Esperados: Tabla Vacia 
df_prestamos_tra.tasa_interes_nominal_anual[df_prestamos_tra.tasa_interes_nominal_anual <= 0]

Series([], Name: tasa_interes_nominal_anual, dtype: float64)

## tasa_interes_efectiva_anual

In [34]:
# Verifica si existen tasas efectivas anuales negativas o igual a 0
# Resultado Esperado: Tabla Vacia 
df_prestamos_tra[df_prestamos_tra.tasa_interes_efectiva_anual <= 0]

,prestamo_id,cliente_id,sucursal_id,producto_id,oficial_id,numero_contrato,fecha_otorgamiento,fecha_vencimiento,monto_original,saldo_capital_vigente,...,estado,dias_mora,clasificacion_riesgo_sbs,garantia_tipo,garantia_valor,proposito_credito,canal_desembolso,fecha_primer_pago_programado,fecha_ultimo_pago_real,dwh_fecha_carga


## Proceso de Revision de Tasas (Tasa nominal y efectiva)

In [35]:
# LEFT JOIN, producto_id 
df_merge_veri_tasas = df_prestamos_tra.merge(
    right=df_productos_crediticios_tra,
    on="producto_id",
    how="left"
)

# Verifica que la tasa nominal este dentro del rango de ese producto 
# Resultados Esperados: Tabla Vacia
error_tasa_nominal = df_merge_veri_tasas[["prestamo_id","tasa_interes_nominal_anual","tasa_interes_efectiva_anual","tasa_nom_min","tasa_nom_max"]][
    (df_merge_veri_tasas.tasa_interes_nominal_anual < df_merge_veri_tasas.tasa_nom_min)
    | (df_merge_veri_tasas.tasa_interes_nominal_anual > df_merge_veri_tasas.tasa_nom_max)
].copy()
error_tasa_nominal

,prestamo_id,tasa_interes_nominal_anual,tasa_interes_efectiva_anual,tasa_nom_min,tasa_nom_max


In [36]:
#==========================================================================================================
# VERIFICA QUE EL CAMBIO ES IGUAL DE AMBOS SENTIDOS, ES DECIR DE EFECTIVA A NOMINAL Y DE NOMINAL A EFECTIVA
#==========================================================================================================

# Selecciona las columnas a utilizar 
tasa_efec = df_prestamos_tra[['prestamo_id','producto_id','tasa_interes_nominal_anual','tasa_interes_efectiva_anual']].copy()

# Recalcula las tasas efectivas utilizando las tasas nominales
tasa_efec["efec_anual_recal"] = tasa_efec.tasa_interes_nominal_anual.apply(
    lambda inte_nominal: 
        round(
            (  
                (
                    (1+(inte_nominal/(12*100))
                    )**12 - 1
                )*100
            ),
            4
        ) 
        if pd.notna(inte_nominal)
        else np.nan
    )

# Realiza un join, con los productos crediticios pero solo con los valores que no coinciden
verificando_tasa_efecti = tasa_efec.merge(
    right=df_productos_crediticios_tra,
    on="producto_id",
    how="left"
)
# Recalculamos la tasa nominal utilizando ahora la tasa efectiva existente 
verificando_tasa_efecti["nomi_anual_recal"] = verificando_tasa_efecti.tasa_interes_efectiva_anual.apply(
    lambda tasa_efectiva: 
        round(   
            (12 * 100*(
                (1 + tasa_efectiva/100)**(1/12)- 1
                )
            ),
            4
        )
        if pd.notna(tasa_efectiva)
        else np.nan
)

# Resultados Esperados: Tabla Vacia
verificando_tasa_efecti[
    ~(verificando_tasa_efecti.tasa_interes_nominal_anual.between(
        left= verificando_tasa_efecti.nomi_anual_recal - 0.03,
        right= verificando_tasa_efecti.nomi_anual_recal +0.03,
        inclusive = 'both'
        )
    )
    |
    ~(verificando_tasa_efecti.tasa_interes_efectiva_anual.between(
        left= verificando_tasa_efecti.efec_anual_recal - 0.03,
        right= verificando_tasa_efecti.efec_anual_recal + 0.03,
        inclusive = 'both'
        )
    )
]

,prestamo_id,producto_id,tasa_interes_nominal_anual,tasa_interes_efectiva_anual,efec_anual_recal,nombre_producto,tipo_credito,tasa_nom_min,tasa_nom_max,plazo_min_meses,plazo_max_meses,monto_minimo,monto_maximo,requiere_garantia,moneda,dwh_fecha_carga,nomi_anual_recal


## plazo_meses

In [37]:
# Verifica si existen plazos de meses negativos o 0
# Resultados Esperados: Tabla Vacia
df_prestamos_tra.plazo_meses[df_prestamos_tra.plazo_meses <= 0]

Series([], Name: plazo_meses, dtype: int64)

## tipo_credito

In [38]:
# Verifica el formato del tipo del credito 
# Resultados Esperados: 'Personal', 'Microempresa', 'Vehicular', 'Hipotecario', 'Consumo'
df_prestamos_tra.tipo_credito.unique()

array(['Personal', 'Microempresa', 'Hipotecario', 'Vehicular', 'Consumo'],
      dtype=object)

In [39]:
# verifica que el tipo de prestamo coincida con el tipo de credito de productos crediticios 
# Resultados Esperados: Tabla Vacia 
verificar_tipo_credito = df_prestamos_tra.merge(
    right=df_productos_crediticios_tra,
    on='producto_id',
    how='left',
    suffixes=["_pres","_produc"]
)

verificar_tipo_credito[["producto_id","tipo_credito_pres","tipo_credito_produc"]][
    verificar_tipo_credito.tipo_credito_pres != verificar_tipo_credito.tipo_credito_produc
    ]

,producto_id,tipo_credito_pres,tipo_credito_produc


## moneda

In [40]:
# Verifica el formato de la modena 
# Resultados Esperados: 'PEN','USD','n/a'
df_prestamos_tra.moneda.unique()

array(['PEN', 'USD'], dtype=object)

## frecuencia_pago

In [41]:
# Resultados Esperados: 'Mensual', 'n/a'
df_prestamos_tra.frecuencia_pago.unique()

array(['Mensual'], dtype=object)

## cuota_programada

In [42]:
# Verifica si existe número negativos o 0 
# Resultado Esperado: Tabla Vacia  
df_prestamos_tra.cuota_programada[df_prestamos_tra.cuota_programada <= 0]

Series([], Name: cuota_programada, dtype: float64)

Para este proyecto se utiliza el sistema frances de creditos, por eso se va a utilizar este sistema para validar que todas las cuota_programada sean validas, la razon, por la que esta verificacion se realiza al final, es porque necesitaba tener, el interes y el numero de periodos. 

prestamo = cuota/TEM * ((1-(1/TEM)**NR_MESES)/(1-(1/TEM)))  
cuota = (prestamo * TEM * (1-(1/TEM))) / (1-(1/TEM)**NR_MESES)  
**Para fines practicos vamos a considerar aceptables todos los valores con margenes de error +-0.03, porque es posible que existan valores diferentes por el momento donde se redondea**

In [43]:
# Selecciona las columnas importantes
verfi_cuota_programada = df_prestamos_tra[['prestamo_id','monto_original','cuota_programada','tasa_interes_efectiva_anual','numero_cuotas_total']].copy()

# Cambia la tasa anual a mensual 
verfi_cuota_programada['tasa_interes_efectiva_mensual'] = verfi_cuota_programada.tasa_interes_efectiva_anual.apply(
    lambda x: ((1+x/100)**(1/12)-1)*100
)

# Recalcula nuevamente la cuota_programada 
verfi_cuota_programada['recal_cuota_programada'] = verfi_cuota_programada.apply(
    axis = 1,
    func= lambda x: 
        round(
            (x['monto_original'] *(1+x['tasa_interes_efectiva_mensual']/100)
                * (1-(1/(1+x['tasa_interes_efectiva_mensual']/100))))
            / (1-(1/(1+x['tasa_interes_efectiva_mensual']/100))**x['numero_cuotas_total']),
            2)
)
# Verifica si existen diferencias entre el recalculo y el valor original
# Resultados Esperados: Tabla Vacia
verfi_cuota_programada[~(verfi_cuota_programada.cuota_programada.between(
    left=verfi_cuota_programada.recal_cuota_programada-0.03,
    right=verfi_cuota_programada.recal_cuota_programada+0.03,
    inclusive='both'
    ))
]

,prestamo_id,monto_original,cuota_programada,tasa_interes_efectiva_anual,numero_cuotas_total,tasa_interes_efectiva_mensual,recal_cuota_programada


## numero_cuotas_total

In [44]:
# Verifica si las cuotas totales son negativas o 0 
# Resultado Esperado: Tabla Vacia
df_prestamos_tra[df_prestamos_tra.numero_cuotas_total <= 0 ]

,prestamo_id,cliente_id,sucursal_id,producto_id,oficial_id,numero_contrato,fecha_otorgamiento,fecha_vencimiento,monto_original,saldo_capital_vigente,...,estado,dias_mora,clasificacion_riesgo_sbs,garantia_tipo,garantia_valor,proposito_credito,canal_desembolso,fecha_primer_pago_programado,fecha_ultimo_pago_real,dwh_fecha_carga


## numero_cuotas_pagadas

In [45]:
# Verifica si las cuotas pagadas son negativas o 0 
# Resultado Esperado: Tabla Vacia
df_prestamos_tra[df_prestamos_tra.numero_cuotas_pagadas <= 0 ]

,prestamo_id,cliente_id,sucursal_id,producto_id,oficial_id,numero_contrato,fecha_otorgamiento,fecha_vencimiento,monto_original,saldo_capital_vigente,...,estado,dias_mora,clasificacion_riesgo_sbs,garantia_tipo,garantia_valor,proposito_credito,canal_desembolso,fecha_primer_pago_programado,fecha_ultimo_pago_real,dwh_fecha_carga


In [46]:
# Verifica si las cuotas pagadas son negativas, 0 o si es mayor a la cantidad total de cuotas
# Resultado Esperado: Tabla Vacia 
df_prestamos_tra[["prestamo_id","numero_cuotas_total","numero_cuotas_pagadas"]][
                (df_prestamos_tra.numero_cuotas_pagadas <= 0) 
                | (df_prestamos_tra.numero_cuotas_pagadas >df_prestamos_tra.numero_cuotas_total)].copy()

,prestamo_id,numero_cuotas_total,numero_cuotas_pagadas


## numero_cuotas_pendientes

In [47]:
# Verifica si las cuotas pendientes son negativas o si son mayores las cuotas totales 
# Resultado Esperado: Tabla Vacia 
df_prestamos_tra[["prestamo_id","numero_cuotas_total","numero_cuotas_pendientes"]][
                (df_prestamos_tra.numero_cuotas_pendientes < 0) 
                | (df_prestamos_tra.numero_cuotas_pendientes >df_prestamos_tra.numero_cuotas_total)].head()

,prestamo_id,numero_cuotas_total,numero_cuotas_pendientes


## Verificar Cuotas (totales, pagas y pendientes)

In [48]:
# Verifica si el numero de cuotas es igual al plazo en meses 
# Resultados Esperados: Tabla Vacia 
df_prestamos_tra[df_prestamos_tra.plazo_meses != df_prestamos_tra.numero_cuotas_total]

,prestamo_id,cliente_id,sucursal_id,producto_id,oficial_id,numero_contrato,fecha_otorgamiento,fecha_vencimiento,monto_original,saldo_capital_vigente,...,estado,dias_mora,clasificacion_riesgo_sbs,garantia_tipo,garantia_valor,proposito_credito,canal_desembolso,fecha_primer_pago_programado,fecha_ultimo_pago_real,dwh_fecha_carga


In [49]:
# Resultados Esperados: Tabla Vacia 
df_prestamos_tra[
    df_prestamos_tra.numero_cuotas_total 
        != (df_prestamos_tra.numero_cuotas_pagadas + df_prestamos_tra.numero_cuotas_pendientes)
]

,prestamo_id,cliente_id,sucursal_id,producto_id,oficial_id,numero_contrato,fecha_otorgamiento,fecha_vencimiento,monto_original,saldo_capital_vigente,...,estado,dias_mora,clasificacion_riesgo_sbs,garantia_tipo,garantia_valor,proposito_credito,canal_desembolso,fecha_primer_pago_programado,fecha_ultimo_pago_real,dwh_fecha_carga


## estado

In [50]:
# Resultados Esperados: 'Vigente', 'Cancelado', 'Moroso', 'Refinanciado', 'Castigado', 'n/a'
df_prestamos_tra.estado.unique()

array(['Vigente', 'Cancelado', 'Moroso', 'Refinanciado'], dtype=object)

##  dias_mora

In [51]:
# Resultados Esperados: Tabla Vacia 
df_prestamos_tra.dias_mora[df_prestamos_tra.dias_mora < 0]

Series([], Name: dias_mora, dtype: int64)

## clasificacion_riesgo_sbs

- **Categoría 0: Normal**  
Créditos de consumo, personal y vehicular: pagos puntuales o hasta 8 días de retraso  
Créditos de microempresa: pagos puntuales o hasta 8 días de retraso  
Créditos hipotecarios: pagos puntuales o hasta 30 días de retraso  
- **Categoría 1: Con problemas potenciales (CPP)**  
Créditos de consumo, personal y vehicular: retrasos entre 9 y 30 días  
Créditos de microempresa: retrasos entre 9 y 30 días  
Créditos hipotecarios: retrasos entre 31 y 60 días  
- **Categoría 2: Deficiente**  
Créditos de consumo, personal y vehicular: retrasos entre 31 y 60 días  
Créditos de microempresa: retrasos entre 31 y 60 días  
Créditos hipotecarios: retrasos entre 61 y 120 días  
- **Categoría 3: Dudoso**  
Créditos de consumo, personal y vehicular: retrasos entre 61 y 120 días  
Créditos de microempresa: retrasos entre 61 y 120 días  
Créditos hipotecarios: retrasos entre 121 y 365 días  
- **Categoría 4: Pérdida**  
Créditos de consumo, personal y vehicular: más de 120 días de atraso  
Créditos de microempresa: más de 120 días de atraso  
Créditos hipotecarios: más de 365 días de atraso  

In [52]:
# Resultados Esperados: 'Normal', 'Dudoso', 'n/a', 'CPP', 'Pérdida', 'Deficiente'
df_prestamos_tra.clasificacion_riesgo_sbs.unique()

array(['Normal', 'CPP', 'Dudoso', 'Deficiente', 'Pérdida'], dtype=object)

In [53]:
recal_riesgo = df_prestamos_tra.apply(
    axis = 1, 
    func = lambda x: formato_clasificacion_riesgo_sbs(x['tipo_credito'],x['dias_mora'])
)

In [54]:
df_prestamos_tra[df_prestamos_tra.clasificacion_riesgo_sbs != recal_riesgo]

,prestamo_id,cliente_id,sucursal_id,producto_id,oficial_id,numero_contrato,fecha_otorgamiento,fecha_vencimiento,monto_original,saldo_capital_vigente,...,estado,dias_mora,clasificacion_riesgo_sbs,garantia_tipo,garantia_valor,proposito_credito,canal_desembolso,fecha_primer_pago_programado,fecha_ultimo_pago_real,dwh_fecha_carga


## garantia_tipo

In [55]:
# Resultados Esperados: 'n/a', 'Aval', 'Carta Fianza', 'Sin Garantía', 'Bien Inmueble', 'Prenda Vehicular', 'Bien Mueble'
df_prestamos_tra.garantia_tipo.unique()

array(['n/a', 'Aval', 'Carta Fianza', 'Sin Garantía', 'Bien Inmueble',
       'Prenda Vehicular', 'Bien Mueble'], dtype=object)

In [56]:
df_merge_verificar_garantia = df_prestamos_tra.merge(
    right=df_productos_crediticios_tra,
    on='producto_id',
    how='left',
)
# Verifica si existan registros que no tengan garantia, cuando el credito solicita que tenga garantia 
# Resultado Esperado: 14 registros
df_revisar_garantia = df_merge_verificar_garantia[['prestamo_id','cliente_id','producto_id','garantia_tipo','requiere_garantia']][(df_merge_verificar_garantia.garantia_tipo == 'n/a')
                & (df_merge_verificar_garantia.requiere_garantia == True)]
df_revisar_garantia

,prestamo_id,cliente_id,producto_id,garantia_tipo,requiere_garantia
243,245,1569,2,n/a,True
442,444,3655,2,n/a,True
1445,1448,1488,2,n/a,True
1654,1657,2383,3,n/a,True
2213,2217,982,3,n/a,True
2479,2483,2467,3,n/a,True
2583,2587,2267,3,n/a,True
3029,3033,210,2,n/a,True
3504,3508,2178,2,n/a,True
3896,3900,2033,3,n/a,True


## garantia_valor

In [57]:
# Verifica que no existan garantias con valor negativo 
# Resultados Esperados: Tabla Vacias 
df_prestamos_tra[df_prestamos_tra.garantia_valor < 0 ]

,prestamo_id,cliente_id,sucursal_id,producto_id,oficial_id,numero_contrato,fecha_otorgamiento,fecha_vencimiento,monto_original,saldo_capital_vigente,...,estado,dias_mora,clasificacion_riesgo_sbs,garantia_tipo,garantia_valor,proposito_credito,canal_desembolso,fecha_primer_pago_programado,fecha_ultimo_pago_real,dwh_fecha_carga


In [58]:
# Cuenta la cantidad de prestamos para cada tipo de credito
df_prestamos_tra.garantia_tipo.value_counts().sort_values(ascending=False)

garantia_tipo
Sin Garantía        1760
Carta Fianza        1741
Aval                1729
Prenda Vehicular     390
Bien Mueble          382
Bien Inmueble        365
n/a                  128
Name: count, dtype: int64

- No estoy muy seguro de como actuar en esta situación, porque, no estoy del todo familiarizado con los terminos.   
- Hasta donde pude indagar, Aval y Carta Fianza si pueden tener un valor, pero por lo que veo en estos datos, parece que no, porque el 100% de este tipo de garantias no tiene un valor.  
- En un entorno real estas consultas se tendrian que resolver con el equipo encargado de ver esta prestamos para que explique la logica a seguir.

In [59]:
# Verifica la cantidad de garantias con valor 0 segun el tipo de garantia
df_prestamos_tra[df_prestamos_tra.garantia_valor == 0 ].groupby("garantia_tipo")["garantia_valor"].size().sort_values(ascending=False)

garantia_tipo
Sin Garantía    1760
Carta Fianza    1741
Aval            1729
n/a              114
Name: garantia_valor, dtype: int64

## proposito_credito

In [60]:
#'Viaje / Turismo', 'Mejoras del Hogar', 'Expansión de Negocio', 'Capital de Trabajo', 'Compra de Vivienda', 'n/a','Compra de Inventario', 'Maquinaria', 'Gastos de Estudios', 'Emergencia Médica', 'Libre Disponibilidad', 'Ampliación y Remodelación', 'Vehículo Nuevo', 'Vehículo Usado', 'Vestimenta y Calzado', 'Electrodomésticos', 'Refinanciamiento Hipotecario', 'Atención Médica', 'Vehículo Comercial', 'Adquisición de Activo Fijo', 'Muebles y Enseres', 'Construcción de Vivienda'
df_prestamos_tra.proposito_credito.unique()

array(['Viaje / Turismo', 'Mejoras del Hogar', 'Expansión de Negocio',
       'Capital de Trabajo', 'Compra de Vivienda', 'n/a',
       'Compra de Inventario', 'Maquinaria', 'Gastos de Estudios',
       'Emergencia Médica', 'Libre Disponibilidad',
       'Ampliación y Remodelación', 'Vehículo Nuevo', 'Vehículo Usado',
       'Vestimenta y Calzado', 'Electrodomésticos',
       'Refinanciamiento Hipotecario', 'Atención Médica',
       'Vehículo Comercial', 'Adquisición de Activo Fijo',
       'Muebles y Enseres', 'Construcción de Vivienda'], dtype=object)

In [61]:
consumo = [
    "Vestimenta y Calzado",
    "Electrodomésticos",
    "Atención Médica",
    "Muebles y Enseres",
    "n/a"
]

personal = [
    "Gastos de Estudios",
    "Viaje / Turismo",
    "Mejoras del Hogar",
    "Emergencia Médica",
    "Libre Disponibilidad",
    "n/a"
]

microempresa = [
    "Expansión de Negocio",
    "Capital de Trabajo",
    "Compra de Inventario",
    "Adquisición de Activo Fijo",
    "n/a"
]

vehicular = [
    "Maquinaria",
    "Vehículo Nuevo",
    "Vehículo Usado",
    "Vehículo Comercial",
    "n/a"
]

hipotecario = [
    "Ampliación y Remodelación",
    "Compra de Vivienda",
    "Construcción de Vivienda",
    "Refinanciamiento Hipotecario",
    "n/a"
]

In [62]:
# Verifica que el proposito del credito coincida con el tipo_credito
# Resultados Esperados: Tabla Vacia
df_prestamos_tra[['prestamo_id','cliente_id','tipo_credito','proposito_credito']][
    ((df_prestamos_tra.tipo_credito == 'Consumo') & ~(df_prestamos_tra.proposito_credito.isin(consumo)))
    | ((df_prestamos_tra.tipo_credito == 'Personal') & ~(df_prestamos_tra.proposito_credito.isin(personal)))
    | ((df_prestamos_tra.tipo_credito == 'Microempresa') & ~(df_prestamos_tra.proposito_credito.isin(microempresa)))
    | ((df_prestamos_tra.tipo_credito == 'Vehicular') & ~(df_prestamos_tra.proposito_credito.isin(vehicular)))
    | ((df_prestamos_tra.tipo_credito == 'Hipotecario') & ~(df_prestamos_tra.proposito_credito.isin(hipotecario)))
]

,prestamo_id,cliente_id,tipo_credito,proposito_credito


## canal_desembolso

In [63]:
# Resultados Esperados: 'Transferencia Bancaria', 'Agencia', 'Cheque de Gerencia', 'n/a'
df_prestamos_tra.canal_desembolso.unique()

array(['Transferencia Bancaria', 'Agencia', 'Cheque de Gerencia'],
      dtype=object)

## fecha_primer_pago_programado

In [64]:
# Muestra las fechas que generan error 
# Resultados Esperados: Tabla Vacia
error_fecha_primer_pag = pd.to_datetime(
    df_prestamos_tra.fecha_primer_pago_programado, 
    errors='coerce'
)
df_prestamos_tra["fecha_primer_pago_programado"][error_fecha_primer_pag.isna()]

Series([], Name: fecha_primer_pago_programado, dtype: datetime64[ns])

In [65]:
# Muestra registros donde la fecha de primer pago sea menor a la fecha de otorgamiento del credito
# Resultados Esperados: Tabla Vacia
df_prestamos_tra[['prestamo_id','fecha_otorgamiento','fecha_primer_pago_programado']][
    df_prestamos_tra.fecha_primer_pago_programado < df_prestamos_tra.fecha_otorgamiento
]

,prestamo_id,fecha_otorgamiento,fecha_primer_pago_programado


In [66]:
# Verifica si existen fechas de pago futuras
# Resultados Esperados: Tabla Vacia
df_prestamos_tra[df_prestamos_tra.fecha_primer_pago_programado.dt.date > date.today()]

,prestamo_id,cliente_id,sucursal_id,producto_id,oficial_id,numero_contrato,fecha_otorgamiento,fecha_vencimiento,monto_original,saldo_capital_vigente,...,estado,dias_mora,clasificacion_riesgo_sbs,garantia_tipo,garantia_valor,proposito_credito,canal_desembolso,fecha_primer_pago_programado,fecha_ultimo_pago_real,dwh_fecha_carga


## fecha_ultimo_pago_real

In [67]:
# Muestra las fechas que generar error 
error_fecha_ultimo_pag = pd.to_datetime(
    df_prestamos_tra.fecha_ultimo_pago_real, 
    errors='coerce'
)
df_prestamos_tra["fecha_ultimo_pago_real"][error_fecha_ultimo_pag.isna()]

Series([], Name: fecha_ultimo_pago_real, dtype: datetime64[ns])

In [68]:
# Fechas ultimo pago real futuras 
# Resultados Esperados: Tabla Vacia
df_prestamos_tra[df_prestamos_tra.fecha_ultimo_pago_real.dt.date >= date.today()]

,prestamo_id,cliente_id,sucursal_id,producto_id,oficial_id,numero_contrato,fecha_otorgamiento,fecha_vencimiento,monto_original,saldo_capital_vigente,...,estado,dias_mora,clasificacion_riesgo_sbs,garantia_tipo,garantia_valor,proposito_credito,canal_desembolso,fecha_primer_pago_programado,fecha_ultimo_pago_real,dwh_fecha_carga


# Segunda Revision de las columnas

## Proceso de Revision de fecha_primer_pago_programado y fecha_otorgamiento

In [69]:
# Los registros siguen esta logica,  fecha_primer_pago_programado == fecha_otorgamiento + 30 dias
# Verifica si esta logica aplica para todos los registros
# Resultadso Esperados: Tabla Vacia
df_prestamos_tra[['prestamo_id','fecha_otorgamiento','fecha_primer_pago_programado']][
    df_prestamos_tra.fecha_primer_pago_programado 
    != df_prestamos_tra.fecha_otorgamiento + pd.DateOffset(days=30)
    ]

,prestamo_id,fecha_otorgamiento,fecha_primer_pago_programado


In [70]:
# Verifica si la fecha de ultimo pago real es menor a la fecha de primer pago programado
df_prestamos_tra[['prestamo_id','fecha_otorgamiento','fecha_primer_pago_programado','fecha_ultimo_pago_real']][df_prestamos_tra.fecha_ultimo_pago_real <= df_prestamos_tra.fecha_primer_pago_programado]

,prestamo_id,fecha_otorgamiento,fecha_primer_pago_programado,fecha_ultimo_pago_real


## Proceso de Revision fecha_vencimiento 

In [71]:
# Recalcula las fechas de vencimiento
fecha_vencimiento_recal = df_prestamos_tra.apply(
    axis = 1,
    func = lambda x: x['fecha_otorgamiento'] + pd.DateOffset(days=30*x['plazo_meses'])
)
# Compara los registros que no cumplan con la logica de fecha vencimiento 
# Resultados Esperados: Tabla Vacia
df_prestamos_tra[['prestamo_id','fecha_otorgamiento','fecha_primer_pago_programado','fecha_vencimiento','plazo_meses']][
    df_prestamos_tra.fecha_vencimiento != fecha_vencimiento_recal
    ]

,prestamo_id,fecha_otorgamiento,fecha_primer_pago_programado,fecha_vencimiento,plazo_meses


## Proceso de Revision del estado del prestamo

Patrones para idenficar el estado del prestamo:  
- **Cancelado**: Siempre que no tenga cuotas pendientes no importa si tiene dias mora.  
- **Vigente**: Siempre que tenga cuotas pendientes y no tenga dias_mora. 
- **Refinanciado**: No encuentro un patron para este estado, porque, no importa si se tiene cuotas o si tiene dias_mora, me imagino que es, porque, el refinanciamiento es algo que decide realizar el cliente no tanto una clasificacion directa del banco, por precaucion, no voy a modificar los estados de los prestamos que tengan estado refinanciado. 
- **Moroso**: Siempre que tenga dias_mora menor a la cantidad necesaria para considerarse **castigado** y tenga cuotas pendientes.  
- **Castigado**: Siempre que tenga días_mora mayor o igual a la cantidad necesaria para considerarse **castigado** y tenga cuotas pendientes.
- **Inconsistente**: Es un tipo de estado que voy asignar cuando no se cumplan ninguna de las otras alternativas posibles.  

In [72]:
recalculo_estado_prestamo = df_prestamos_tra.apply(
    axis = 1,
    func = lambda x: asignacion_estado_prestamo(x["numero_cuotas_pendientes"],x["dias_mora"],x["estado"],x["tipo_credito"])
)
df_prestamos_tra[df_prestamos_tra.estado != recalculo_estado_prestamo]

,prestamo_id,cliente_id,sucursal_id,producto_id,oficial_id,numero_contrato,fecha_otorgamiento,fecha_vencimiento,monto_original,saldo_capital_vigente,...,estado,dias_mora,clasificacion_riesgo_sbs,garantia_tipo,garantia_valor,proposito_credito,canal_desembolso,fecha_primer_pago_programado,fecha_ultimo_pago_real,dwh_fecha_carga


## Proceso de Revision fecha_otorgamiento V2

### fecha_registro (relacion con clientes)

In [73]:
# LEFT JOIN, por cliente_id 
df_veri_fecha_presta = df_prestamos_tra[['prestamo_id','cliente_id','fecha_otorgamiento']].copy()
df_veri_fecha_clie = df_clientes_tra[['cliente_id','fecha_registro']]
df_merge_verificar_fecha_otor = df_veri_fecha_presta.merge(
    right= df_veri_fecha_clie,
    on="cliente_id",
    how='left'
)
# Verifica que la fecha_otorgamiento no sea mayor a la fecha_registro 
# Resultados Esperados: Tabla Vacia 
df_merge_verificar_fecha_otor[
    (df_merge_verificar_fecha_otor.fecha_otorgamiento 
        < df_merge_verificar_fecha_otor.fecha_registro)
]

,prestamo_id,cliente_id,fecha_otorgamiento,fecha_registro


### fecha_apertura (relacion con sucursal)

In [74]:
# LEFT JOIN, por sucursal_id 
df_veri_fecha_apertu = df_prestamos_tra[['prestamo_id','sucursal_id','fecha_otorgamiento']].copy()
df_veri_fecha_sucur = df_sucursales_tra.copy()
df_merge_veri_fecha_apertu = df_veri_fecha_apertu.merge(
    right= df_veri_fecha_sucur,
    on="sucursal_id",
    how='left'
)
# Verifica que la fecha_otorgamiento no sea mayor a la fecha_apertura 
# Resultados Esperados: Tabla Vacia 
df_merge_veri_fecha_apertu[
    (df_merge_veri_fecha_apertu.fecha_otorgamiento 
        < df_merge_veri_fecha_apertu.fecha_apertura)
]

,prestamo_id,sucursal_id,fecha_otorgamiento,fecha_apertura


### fecha_ingreso (relacion con oficial de credito)

In [75]:
# LEFT JOIN, por oficial_id 
df_veri_fecha_ingreso = df_prestamos_tra[['prestamo_id','oficial_id','fecha_otorgamiento']].copy()
df_veri_fecha_ingreso_ofi = df_oficial_tra.copy()
df_merge_veri_fecha_ingreso = df_veri_fecha_ingreso.merge(
    right= df_veri_fecha_ingreso_ofi,
    on="oficial_id",
    how='left'
)
# Verifica que la fecha_otorgamiento no sea mayor a la fecha_apertura 
# Resultados Esperados: Tabla Vacia 
df_merge_veri_fecha_ingreso[
    (df_merge_veri_fecha_ingreso.fecha_otorgamiento 
        < df_merge_veri_fecha_ingreso.fecha_ingreso)
]

,prestamo_id,oficial_id,fecha_otorgamiento,fecha_ingreso
